# Setup

Runs on **Google Colab** or **locally** (`COLAB_RELEASE_TAG` is used to detect Colab). Paths, Drive mount, clone, and dataset unzip run only when needed.

In [1]:
import os
from pathlib import Path
import yaml
import subprocess
import sys
from datetime import datetime
import zipfile
import copy

IN_COLAB = bool(os.environ.get("COLAB_RELEASE_TAG"))
REPO_URL = "https://github.com/chendwend/thesis-assyrian-relief.git"
TIMESTAMP = datetime.now().strftime("%d-%m_%H-%M-%S")


def find_repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "configs" / "style_dinov2.yaml").is_file():
            return candidate
    raise FileNotFoundError(
        "Cannot find repo root (missing configs/style_dinov2.yaml). "
        "Cd to the thesis-assyrian-relief clone or open the notebook from that repo."
    )


def sync_notebook_cwd(path: Path) -> None:
    """Match Python and IPython shell cwd so `!command` cells run in the repo."""
    os.chdir(path)
    try:
        from IPython import get_ipython

        ip = get_ipython()
        if ip is not None:
            ip.run_line_magic("cd", str(path))
    except ImportError:
        pass


if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    REPO_DIR = Path("/content/thesis-assyrian-relief")
    DRIVE_ROOT = Path("/content/drive/MyDrive/Graduate_Studies/Thesis")
    IMAGE_ROOT = Path("/content/dataset")
    
    OUTPUT_ROOT = DRIVE_ROOT / "outputs" / TIMESTAMP
else:
    REPO_DIR = find_repo_root()
    sync_notebook_cwd(REPO_DIR)
    import yaml

    with open(REPO_DIR / "configs" / "style_dinov2.yaml", encoding="utf-8") as f:
        _model_cfg = yaml.safe_load(f)

        
    IMAGE_ROOT = Path(_model_cfg["data"]["image_root"]).expanduser()
    OUTPUT_ROOT = (REPO_DIR / "outputs"/TIMESTAMP).resolve()



# Unzip dataset if in Colab
if IN_COLAB:
    for zip_path in (Path("/content/dataset.zip"), Path.cwd() / "dataset.zip"):
        if zip_path.is_file():
            with zipfile.ZipFile(zip_path) as zf:
                zf.extractall(Path("/content"))
            zip_path.unlink(missing_ok=True)
            print(f"Extracted and removed {zip_path}")
            break
    else:
        print("No dataset.zip found; skip unzip (use Drive-mounted data or upload zip).")
else:
    print("Local: skip dataset unzip.")


# Clone repo if in Colab
if IN_COLAB:
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], cwd="/content", check=True)
    sync_notebook_cwd(REPO_DIR)
else:
    if not (REPO_DIR / "configs" / "style_dinov2.yaml").is_file():
        raise FileNotFoundError(REPO_DIR / "configs" / "style_dinov2.yaml")

print("CWD:", Path.cwd())

print(f"IN_COLAB={IN_COLAB}")
print(f"REPO_DIR={REPO_DIR}")
print(f"IMAGE_ROOT={IMAGE_ROOT}")
print(f"OUTPUT_ROOT={OUTPUT_ROOT}")

/home/kostya/projects/thesis-assyrian-relief
Local: skip dataset unzip.
CWD: /home/kostya/projects/thesis-assyrian-relief
IN_COLAB=False
REPO_DIR=/home/kostya/projects/thesis-assyrian-relief
IMAGE_ROOT=/mnt/c/Users/chend/Desktop/My_Files/Thesis/Dataset/dataset_v2
OUTPUT_ROOT=/home/kostya/projects/thesis-assyrian-relief/outputs/16-05_09-49-05


In [2]:
OUTPUT_ROOT = OUTPUT_ROOT.with_name("only_3_classes")

In [ ]:
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run(["uv", "sync"], cwd=str(REPO_DIR), check=True)

# Set Runtime configs

In [20]:
base_cfg_path = Path("configs/style_dinov2.yaml")
runtime_cfg_path = Path(f"{OUTPUT_ROOT}/style_dinov2_runtime.yaml")

with open(base_cfg_path, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

cfg["data"]["image_root"] = str(IMAGE_ROOT)

cfg["outputs"] = {
    "checkpoint_path": str(OUTPUT_ROOT / "checkpoints" / "dinov2_probe.pt"),
    "history_path": str(OUTPUT_ROOT / "checkpoints" / "dinov2_probe.history.csv"),
    "training_curve_path": str(OUTPUT_ROOT / "plots" / "train_curves.png"),

    "umap_html_path": str(OUTPUT_ROOT / "plots" / "umap_plot.html"),
    "umap_csv_path": str(OUTPUT_ROOT / "plots" / "umap_plot.csv"),

    "retrieval_metrics_path": str(OUTPUT_ROOT / "retrieval" / "test_metrics.json"),
    "retrieval_top1_path": str(OUTPUT_ROOT / "retrieval" / "test_top1.csv"),
    "retrieval_topk_path": str(OUTPUT_ROOT / "retrieval" / "test_topk.csv"),
    "retrieval_failures_path": str(OUTPUT_ROOT / "retrieval" / "test_failures.csv"),

    # Evaluation outputs by split
    "eval": {
        "val": {
            "confusion_matrix_path": str(OUTPUT_ROOT / "plots" / "confusion_matrix_val.png"),
            "eval_metrics_path": str(OUTPUT_ROOT / "eval" / "val_metrics.json"),
            "eval_retrieval_path": str(OUTPUT_ROOT / "eval" / "val_retrieval.csv"),
            "relief_predictions_path": str(OUTPUT_ROOT / "eval" / "val_relief_predictions.csv"),
        },
        "test": {
            "confusion_matrix_path": str(OUTPUT_ROOT / "plots" / "confusion_matrix_test.png"),
            "eval_metrics_path": str(OUTPUT_ROOT / "eval" / "test_metrics.json"),
            "eval_retrieval_path": str(OUTPUT_ROOT / "eval" / "test_retrieval.csv"),
            "relief_predictions_path": str(OUTPUT_ROOT / "eval" / "test_relief_predictions.csv"),
        },
    },
}

cfg["evaluation"] = {
    "relief_aggregation": "mean_logits",
}

cfg["umap"] = {
    "fit_split": "train",
    "plot_splits": ["train", "val", "test"],
    "highlight_relief_ids": [],
}

def iter_output_paths(obj):
    if isinstance(obj, dict):
        for v in obj.values():
            yield from iter_output_paths(v)
    elif isinstance(obj, str):
        yield obj

for out_path in iter_output_paths(cfg["outputs"]):
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)

with open(runtime_cfg_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print(f"Saved runtime config to: {runtime_cfg_path}")

Saved runtime config to: /home/kostya/projects/thesis-assyrian-relief/outputs/only_3_classes/style_dinov2_runtime.yaml


# Training

In [5]:
!uv run python scripts/train_style.py \
  --config {runtime_cfg_path}

Resolved config:
  csv_path: data/splits/image_level_dataset_v2.csv
  image_root: /mnt/c/Users/chend/Desktop/My_Files/Thesis/Dataset/dataset_v2
  filename_sep: -
  train_split: train
  val_split: val
  model_name: dinov2_vits14
  emb_dim: 256
  batch_size: 16
  num_workers: 2
  num_epochs: 50
  optimizer_name: adamw
  lr: 0.001
  weight_decay: 0.0001
  loss_name: cross_entropy
  class_weighting: balanced
  scheduler_enabled: True
  scheduler_name: reduce_on_plateau
  scheduler_mode: max
  scheduler_factor: 0.5
  scheduler_patience: 2
  scheduler_monitor: val_macro_f1
  checkpoint_monitor: val_macro_f1
  checkpoint_mode: max
  checkpoint_path: /home/kostya/projects/thesis-assyrian-relief/outputs/15-05_11-33-45/checkpoints/dinov2_probe.pt
  history_path: /home/kostya/projects/thesis-assyrian-relief/outputs/15-05_11-33-45/checkpoints/dinov2_probe.history.csv
  training_curve_path: /home/kostya/projects/thesis-assyrian-relief/outputs/15-05_11-33-45/plots/train_curves.png
  early_stopping_e

# Validation

In [ ]:
!uv run python scripts/eval_style.py \
  --config {runtime_cfg_path} \
  --eval-split val

Resolved config:
  csv_path: data/splits/image_level_dataset_v2.csv
  image_root: /mnt/c/Users/chend/Desktop/My_Files/Thesis/Dataset/dataset_v2
  filename_sep: -
  train_split: train
  eval_split: val
  checkpoint_path: /home/kostya/projects/thesis-assyrian-relief/outputs/15-05_11-33-45/checkpoints/dinov2_probe.pt
  batch_size: 16
  num_workers: 2
  eval_loss_class_weighting: none
  metrics_out: /home/kostya/projects/thesis-assyrian-relief/outputs/15-05_11-33-45/eval/val_metrics.json
  retrieval_out: /home/kostya/projects/thesis-assyrian-relief/outputs/15-05_11-33-45/eval/val_retrieval.csv
  confusion_matrix_path: /home/kostya/projects/thesis-assyrian-relief/outputs/15-05_11-33-45/plots/confusion_matrix_val.png
  relief_aggregation: mean_logits
  relief_preds_out: /home/kostya/projects/thesis-assyrian-relief/outputs/15-05_11-33-45/eval/val_relief_predictions.csv
class_to_idx: {'Ashurbanipal': 0, 'Ashurnasirpal II': 1, 'Sargon II': 2}
model_name: dinov2_vits14
emb_dim: 256
Using device:

# Evaluation

In [ ]:
!uv run python scripts/eval_style.py \
  --config {runtime_cfg_path} \
  --eval-split test

# UMAP

In [21]:
!uv run python scripts/umap_style.py \
  --config {runtime_cfg_path}

Resolved config:
  csv_path: data/splits/image_level_dataset_v2.csv
  image_root: /mnt/c/Users/chend/Desktop/My_Files/Thesis/Dataset/dataset_v2
  filename_sep: -
  train_split: train
  eval_split: test
  checkpoint_path: /home/kostya/projects/thesis-assyrian-relief/outputs/only_3_classes/checkpoints/dinov2_probe.pt
  batch_size: 16
  num_workers: 2
  html_out: /home/kostya/projects/thesis-assyrian-relief/outputs/only_3_classes/plots/umap_plot.html
  csv_out: /home/kostya/projects/thesis-assyrian-relief/outputs/only_3_classes/plots/umap_plot.csv
  highlight_relief_ids: []
  umap_fit_split: train
  umap_plot_splits: ['train', 'val', 'test']
Using device: cuda
class_to_idx: {'Ashurbanipal': 0, 'Ashurnasirpal II': 1, 'Sargon II': 2}
Using cache found in /home/kostya/.cache/torch/hub/facebookresearch_dinov2_main
Extracting embeddings for split='test'...
Extracting embeddings for split='train'...
Extracting embeddings for split='val'...
Saved interactive UMAP HTML to: /home/kostya/projects/t

# Retrieval Analysis

In [21]:
!uv run python scripts/retrieval_analysis.py \
  --config configs/style_dinov2_runtime.yaml \
  --eval-split test

Resolved config:
  csv_path: data/splits/image_level_dataset_v2.csv
  image_root: /mnt/c/Users/chend/Desktop/My_Files/Thesis/Dataset/dataset_v2
  filename_sep: -
  train_split: train
  eval_split: test
  checkpoint_path: /home/kostya/projects/thesis-assyrian-relief/outputs/checkpoints/dinov2_probe_v2.pt
  batch_size: 16
  num_workers: 0
  topk: 5
  metrics_out: /home/kostya/projects/thesis-assyrian-relief/outputs/retrieval/test_metrics_v2.json
  top1_out: /home/kostya/projects/thesis-assyrian-relief/outputs/retrieval/test_top1_v2.csv
  topk_out: /home/kostya/projects/thesis-assyrian-relief/outputs/retrieval/test_topk_v2.csv
  failures_out: /home/kostya/projects/thesis-assyrian-relief/outputs/retrieval/test_failures_v2.csv
Using device: cuda
class_to_idx: {'Ashurbanipal': 0, 'Ashurnasirpal II': 1, 'Sargon II': 2, 'Sennacherib': 3, 'Tiglath-Pileser III': 4}
Using cache found in /home/kostya/.cache/torch/hub/facebookresearch_dinov2_main
Extracting train embeddings...
Extracting eval embed

## test

In [ ]:
!uv run python scripts/train_style.py \
  --config configs/style_dinov2_runtime.yaml \
  --num-epochs 1 \
  --batch-size 8 \
  --num-workers 2